# Reproduce everything

One traceable run of the full evidentiary chain: environment report, test
suite, the two calibration studies behind the paper's headline claims, the
comparison benchmarks (real datasets pulled live from their sources), and
the physics panel. Where a competitor cannot run in a modern environment
(autofeat's dependency pins) or the evidence is an external study, the
archived artifact is loaded and **labelled as archived** — traceability
includes being explicit about what runs live and what is replayed.

Runtime: roughly 6–8 minutes. Requires `pip install -e ".[all]"` from the
repository root and network access for the seaborn-data CSVs.

In [1]:
import json, pathlib, platform, subprocess, sys
import numpy, sklearn
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "benchmarks" else pathlib.Path.cwd()
print("python ", sys.version.split()[0], "|", platform.platform())
print("numpy  ", numpy.__version__, "| scikit-learn", sklearn.__version__)

python  3.12.3 | Linux-6.18.5-x86_64-with-glibc2.39
numpy   2.5.1 | scikit-learn 1.8.0


## 1. Test suite

The fast suite; the three `slow`-marked estimator-conformance
tests run in CI on every push and are excluded here only for runtime.

In [2]:
run = subprocess.run([sys.executable, "-m", "pytest", "-q", "-m", "not slow", str(ROOT / "tests")],
                     capture_output=True, text=True, cwd=ROOT)
print(run.stdout.strip().splitlines()[-1])
assert run.returncode == 0, "test suite failed"


367 passed, 3 deselected, 1 warning in 37.48s


## 2. Calibration at publication scale

Regenerates the paper's headline: 200 signal replicates at nominal FDR
0.10, then 60 global-null replicates (`benchmarks/calibration_study.py`).

In [3]:
sys.path.insert(0, str(ROOT / "benchmarks"))
import calibration_study
calibration = calibration_study.main()
assert calibration["empirical_fdr"] == 0.0 and calibration["power"] == 1.0


SIGNAL (200 replicates, nominal 0.10): empirical FDR 0.0000 | power 1.000 | fallbacks 0
GLOBAL NULL (60 replicates): selections 0


## 3. Friedman #1 decomposition

The measured boundary of the marginal guarantee
(`benchmarks/friedman_decomposition.py`).

In [4]:
import friedman_decomposition
decomposition = friedman_decomposition.main()
assert round(decomposition["oracle"], 3) == 0.964
assert round(decomposition["ceiling"], 3) == 0.875


representation oracle (all six terms):        R^2 0.964
screening-admissible ceiling (BY admits ['ab', '(ab)^2', 'd', 'e']): R^2 0.875
default pipeline on this split:               R^2 0.784


## 4. Comparison benchmarks

The committed harness: core, stress, scenarios, and the six real datasets —
seaborn CSVs fetched live from `raw.githubusercontent.com/mwaskom/seaborn-data`,
so provenance is the source itself. Methods here: beamfeat, ridge, LightGBM;
knockpy and OpenFE rows regenerate via `--methods knockpy` / `openfe` when
installed, and their archived rows sit in `results_knockpy*.json` and
`results_robustness.json`.

In [5]:
for args in (["--suite", "all", "--synthetic-only"], ["--real-only"]):
    run = subprocess.run([sys.executable, str(ROOT / "benchmarks" / "run_benchmarks.py"),
                          *args, "--methods", "ridge", "lightgbm", "beamfeat",
                          "--output", "/tmp/reproduce.csv"],
                         capture_output=True, text=True, cwd=ROOT)
    tail = [l for l in run.stdout.splitlines() if "beamfeat:" in l or "recovered" in l]
    print(chr(10).join(tail[-6:]), chr(10))


    beamfeat: R2  0.8080     0.40s    9 feats
    beamfeat: R2  0.9858     0.31s    2 feats  [recovered]
    beamfeat: R2  0.9906     0.31s    2 feats  [recovered]
    beamfeat: 15/15 recovered
    beamfeat: mean  0.000   worst 0.000 (product)
    beamfeat:   2.03 



    beamfeat: R2  0.9649     0.98s   11 feats
    beamfeat: R2  0.8708     0.30s   23 feats
    beamfeat: R2  0.8708     0.13s    1 feats
    beamfeat: R2  0.7442     0.05s    7 feats
    beamfeat: R2  0.4962     0.05s   15 feats
    beamfeat:   2.74 



## 5. Physics equation panel

In [6]:
run = subprocess.run([sys.executable, str(ROOT / "benchmarks" / "feynman_panel.py")],
                     capture_output=True, text=True, cwd=ROOT)
print(chr(10).join(l for l in run.stdout.splitlines() if l.startswith("[")))


[default operators]  solved 10/12 (R^2>0.999)  exact-form 8  mean 0.6s/eq
[with exp enabled]  solved 10/12 (R^2>0.999)  exact-form 8  mean 0.5s/eq


## 6. Archived competitor evidence

autofeat cannot execute here — its pins and the scikit-learn ≥ 1.8
transform failure require the isolated environment described in
`benchmarks/independent/requirements.txt` — so its rows load from archived
artifacts produced there, and the independent 315-fit study loads from its
as-reported results.

In [7]:
import pandas as pd
autofeat = pd.DataFrame(json.load(open(ROOT / "benchmarks" / "results_autofeat_venv.json")))
print("[archived] autofeat core: mean R2 %.4f, %d/%d recovered, %.1fs mean"
      % (autofeat.r2.mean(), int(autofeat.recovered.sum()),
         int(autofeat.recovered.notna().sum()), autofeat.fit_seconds.mean()))
study = pd.read_csv(ROOT / "benchmarks" / "independent" / "results_as_reported" / "independent_benchmark_results.csv")
agg = study.groupby("method").r2.agg(["mean", "min"]).round(3).sort_values("mean", ascending=False)
print("[archived] independent 315-fit study (mean / worst R2):")
print(agg.to_string())


[archived] autofeat core: mean R2 0.9974, 8/9 recovered, 9.0s mean


[archived] independent 315-fit study (mean / worst R2):
               mean     min
method                     
rf_raw        0.810   0.247
beamfeat      0.803   0.355
lgbm_raw      0.798   0.117
autofeat      0.751  -2.282
openfe        0.736   0.360
ridge_raw     0.704   0.353
featuretools -2.478 -57.320


Every number above either regenerated live in this session or was read
from a committed artifact whose producing command is recorded in
`benchmarks/README.md`.